In [4]:
Q1 = "I just discovered the course, can I still join?"

In [5]:
Q2 = "I just found out about the program, can I still enroll?"

In [6]:
# important words: discover, course, join 
# important words: find, program, enroll

In [7]:
from sentence_transformers import SentenceTransformer

In [8]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
q1 = "Can I still join the course after the start date?"

In [10]:
v1 = model.encode(q1)

In [11]:
print(len(v1))

384


In [12]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [13]:
v1.dot(dv)

np.float32(0.32332397)

In [14]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [15]:
v2.dot(dv)

np.float32(0.019730438)

In [16]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-08-11 03:39:09--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py.1’

ingest.py.1         100%[===================>]     738  --.-KB/s    in 0s      

2026-08-11 03:39:09 (35.1 MB/s) - ‘ingest.py.1’ saved [738/738]



In [17]:
from ingest import load_faq_data

In [18]:
documents = load_faq_data()

In [19]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [20]:
len(texts)

1406

In [21]:
from tqdm import tqdm
from time import sleep

nombres = ["Ana", "Luis", "Pedro", "María"]

for nombre in tqdm(nombres):
    sleep(1)
    print(nombre)

 25%|████████████████████████████                                                                                    | 1/4 [00:01<00:03,  1.00s/it]

Ana


 50%|████████████████████████████████████████████████████████                                                        | 2/4 [00:02<00:02,  1.00s/it]

Luis


 75%|████████████████████████████████████████████████████████████████████████████████████                            | 3/4 [00:03<00:01,  1.00s/it]

Pedro


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.00s/it]

María


In [22]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
len(vectors)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29/29 [01:42<00:00,  3.52s/it]


1406

In [23]:
# print(vectors[10]) # print one element
v1.dot(vectors[10]) # dot product for q1 to compare similarity

np.float32(0.33153275)

In [24]:
# better to work with arrays 
import numpy as np
X = np.array(vectors)

In [25]:
X.shape

(1406, 384)

In [26]:
# one simple way to do it, it's iterate over vectors comparing 
scores = []
for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [27]:
new_scores = X.dot(v1)

In [28]:
new_scores

array([0.48740575, 0.20991933, 0.762941  , ..., 0.2028995 , 0.13746649,
       0.09721461], shape=(1406,), dtype=float32)

In [31]:
# argmax returns index with max
idx = np.argmax(new_scores)
print(idx)

2


In [32]:
scores[idx]

np.float32(0.76294106)

In [34]:
print(v1[:10])

[ 0.02139041 -0.07397997  0.00142069  0.02138166  0.02451131  0.03155828
 -0.1108397  -0.1050175  -0.06182589 -0.00642312]


In [35]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [41]:
# np.argsort
top5 = np.argsort(new_scores)[-5:]
top5 = top5[::-1]

In [43]:
new_scores[top5]


array([0.762941  , 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

In [44]:
for idx in top5:
    print(idx)
    print(documents[idx])

2
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}
1155
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}
567
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'questio

In [46]:
for idx in np.argsort(-new_scores)[:5]:
    print(idx)
    print(documents[idx])

2
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}
1155
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}
567
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'questio